<br><h1>Fake News Detection Part 2</h1><br>

In [2]:
# libraries for part 1
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
import re
import sys

pd.set_option('display.max_columns', 200)
plt.style.use("ggplot")

# Libraries for part 2
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler

In [3]:
# read in data
df = pl.read_csv("./data/995,000_rows_preprocessed.csv", n_rows=10000).to_pandas()
df.head()

,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,meta_keywords,meta_description,tags,source
0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,plus one articl googl plus thank ali alfoneh a...,2017-11-27T01:14:42.983556,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,iran news round,nan,nation review nation review onlin articl,nan,nan,nan
1,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,cost best senat bank committe jp morgan buy <N...,2017-11-27T01:14:08.7454,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,cost best senat bank committe jp morgan buy <N...,nan,nan,nan,nan,nan
2,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,man awoken <NUM> -year coma commit suicid lear...,2017-11-27T01:14:21.395055,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,man awoken <NUM> -year coma commit suicid lear...,nan,nan,nan,nan,nan
3,nytimes.com,reliable,https://query.nytimes.com/gst/fullpage.html?re...,julia geist ask draw pictur comput scientist l...,2018-02-11 00:46:42.632962,2018-02-11 00:14:20.346838,2018-02-11 00:14:20.346871,open gateway girl enter comput field,nan,comput internet women girl career profess mill...,julia geist ask draw pictur comput scientist l...,nan,nytimes
4,infiniteunknown.net,conspiracy,http://www.infiniteunknown.net/2011/09/14/100-...,<NUM> compil studi vaccin danger activist post...,2017-11-10T11:18:44.524042,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,<NUM> compil studi vaccin danger infinit unknown,nan,nan,nan,"Lymphoma, Hepatitis B, Immune System, Health, ...",nan


In [10]:
# Unique labels
print(df['type'].unique())

# We choose to drop the following
drop_rows = ['unknown', 'nan', '2018-02-10 13:43:39.521661',
             'satire', 'bias', 'clickbait', 'unreliable', 'state', 'nan']

# Drop all rows of df with label
dropped = df.drop(df.loc[df['type'].isin(drop_rows)].index)

# Split dataframe in feature and label without loss of information 
# (further reduction needed).
y = dropped['type'].to_list()
X = dropped.drop(['type'], axis=1)
# Split into train set (80%), validation set (10%), and test set(10%)
X_train_raw, X_valtest, y_train_raw, y_valtest = train_test_split(
    X, y, test_size=0.2, random_state=1000
)
X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(
    X_valtest, y_valtest, test_size=0.5, random_state=1001
)

<ArrowStringArray>
[ 'political',       'fake',     'satire',   'reliable', 'conspiracy',
 'unreliable',       'bias',      'rumor',    'unknown',        'nan',
  'clickbait',       'hate',    'junksci']
Length: 13, dtype: str


### Task 1 - grouping the labels
To group the labels in two groups, we first identified the unique labels used. This also guarantees that the drop before went as wished.

In [11]:
print(' '.join(sorted(list(set(y)))))

conspiracy fake hate junksci political reliable rumor


In [17]:
# Group labels in two groups. This is partly subjective which might be a problem later.
# We do this by specifying reliable labels and let all other labels be defined as fake.
# real_labs = ['political', 'satire', 'reliable', 'bias']
real_labs = ['reliable', 'political']

raw_ys = [y_train_raw, y_val_raw, y_test_raw]
y_train, y_val, y_test = [], [], []
ys = [y_train, y_val, y_test]

# We now transform the ys to be binary: 1 for fake, 0 for real.
for i, raw_y in enumerate(raw_ys):
    for j, lab in enumerate(raw_y):
        if lab in real_labs:
            ys[i].append(0)
        else:
            ys[i].append(1)

pd.concat([pd.Series(y_train_raw, name='label'), pd.Series(y_train, name='fake')], axis=1).head(20)

,label,fake
0,fake,1
1,political,0
2,conspiracy,1
3,conspiracy,1
4,reliable,0
5,hate,1
6,fake,1
7,political,0
8,reliable,0
9,political,0


### Task 2
To implement the logistic regression, we needed to extract the features from the training and test data. However, we ran into a problem.

If we used a traditional pandas or polars DataFrame, we would use too much RAM. We have 995,000 rows and using the top 10,000 words of the vocabulary with standard Int64 representation (8 bytes), this would take

995,000 * 10,000 * 8 bytes 
= 9.95 * 8 * 10^9 bytes 
= 79.6 * 10^9 bytes
= 79.6 GB

of RAM. This is absurd. However, we could expect that most of the BoW data are just zeros. Therefore, we might represent it using a sparse matrix as fx. a Compressed Sparse Row (CSR) matrix. This can be done efficiently using the CountVectorizer from sklearn.

In [18]:
# Given a pandas or polars dataframe and potentially an number of tokens 
# to be included as features, returns a bag of words representation of
# the input's column 'content' as a scipy.sparse.csr_matrix.
def bow(df: pd.DataFrame | pl.DataFrame, 
        num_feats = 10**4,
        save_as: None | str = None):

    # Create BoW using the CountVectorizer module
    # Define corpus using polars for efficiency
    if type(df) == pd.DataFrame:
        pl_series = pl.from_pandas(df['content'])
    else:
        pl_series = df.select('content')

    # Concat natural language cols to single list
    docs = pl_series.to_list()

    # Initialize vectorizer
    vec = CountVectorizer(max_features=num_feats)

    # Fit vectorizer to create compressed sparse column matrix for RAM efficiency
    X = vec.fit_transform(docs)

    if save_as: scipy.sparse.save_npz(save_as, X)
    
    return X

In [19]:
# Create BoW representation of trainig, validation and test set
X_train = bow(X_train_raw, save_as='./data/X_train.npz')
X_val = bow(X_val_raw, save_as='./data/X_val.npz')
X_test = bow(X_test_raw, save_as='./data/X_test.npz')

We now implement the logistic regression and evaluate it with F1 score.

In [20]:
scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_val = scaler.transform(X_val)

classifier = LogisticRegression(random_state=1000, max_iter=2500)
classifier.fit(X_train, y_train)

preds = classifier.predict(X_val)
score = f1_score(y_val, preds)

print(f'F1 Score: {score:.3f}')

F1 Score: 0.418
